# 04 — Model Training

**Project:** Predictive Analytics for Enterprise Streaming Acquisitions  
**Course:** CMPS 451 — Data Mining, Big Data & Analytics (Spring 2026)  
**Team:** 11

---

## Objectives
1. Build regression models (Linear, Decision Tree, Random Forest, GBT) to predict `averageRating`
2. Build classification models (Logistic, Decision Tree, Random Forest) to predict rating tier
3. Apply K-Means clustering for title segmentation
4. Perform hyperparameter tuning with CrossValidator
5. Save all trained models

In [ ]:
# ── Fix PySpark path issue with spaces ──
import os
import sys
venv_scripts = os.path.dirname(sys.executable)
os.environ["PATH"] = f"{venv_scripts};{os.environ.get('PATH', '')}"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

# ── Imports ──
import os
import json
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import (
    LinearRegression, DecisionTreeRegressor,
    RandomForestRegressor, GBTRegressor
)
from pyspark.ml.classification import (
    LogisticRegression, DecisionTreeClassifier,
    RandomForestClassifier
)
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    RegressionEvaluator, MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

BASE_DIR = r"e:\CUFE\Spring 25\Big Data\Project"
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Fix Windows PySpark: set HADOOP_HOME
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
# Fix Windows PySpark: set HADOOP_HOME
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

    SparkSession.builder
    .appName("IMDb_Modeling")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready.")

In [ ]:
# ── Load feature matrix ──
df = spark.read.parquet(os.path.join(OUTPUT_DIR, "parquet", "features"))

with open(os.path.join(OUTPUT_DIR, "feature_columns.json")) as f:
    feat_meta = json.load(f)

feature_cols = feat_meta['all_feature_cols']
print(f"Loaded {df.count():,} rows with {len(feature_cols)} features")

# Drop rows with any null in feature columns
df = df.dropna(subset=feature_cols + ['averageRating', 'ratingTier'])
print(f"After dropping nulls: {df.count():,} rows")

In [ ]:
# ── Assemble feature vector ──
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw", handleInvalid="skip")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

pipeline = Pipeline(stages=[assembler, scaler])
pipeline_model = pipeline.fit(df)
df_ml = pipeline_model.transform(df)

# Rename targets
df_ml = df_ml.withColumnRenamed("averageRating", "label_reg")
df_ml = df_ml.withColumnRenamed("ratingTier", "label_cls")

print(f"Feature vector assembled: {len(feature_cols)} features")
df_ml.select("features", "label_reg", "label_cls").show(3, truncate=50)

In [ ]:
# ── Train / Validation / Test Split (70/15/15) ──
train_df, val_df, test_df = df_ml.randomSplit([0.70, 0.15, 0.15], seed=42)

train_df.cache()
val_df.cache()
test_df.cache()

print(f"Train: {train_df.count():,}")
print(f"Validation: {val_df.count():,}")
print(f"Test: {test_df.count():,}")

## 1. Regression Models (Predict `averageRating`)

In [ ]:
# ── Helper function ──
reg_evaluator_rmse = RegressionEvaluator(labelCol="label_reg", metricName="rmse")
reg_evaluator_mae = RegressionEvaluator(labelCol="label_reg", metricName="mae")
reg_evaluator_r2 = RegressionEvaluator(labelCol="label_reg", metricName="r2")

regression_results = []

def eval_regression(model_name, model, train, val, test):
    results = {"model": model_name}
    for split_name, data in [("train", train), ("val", val), ("test", test)]:
        preds = model.transform(data)
        rmse = reg_evaluator_rmse.evaluate(preds)
        mae = reg_evaluator_mae.evaluate(preds)
        r2 = reg_evaluator_r2.evaluate(preds)
        results[f"{split_name}_rmse"] = round(rmse, 4)
        results[f"{split_name}_mae"] = round(mae, 4)
        results[f"{split_name}_r2"] = round(r2, 4)
    regression_results.append(results)
    print(f"\n{model_name}:")
    print(f"  Train  — RMSE: {results['train_rmse']:.4f}, MAE: {results['train_mae']:.4f}, R²: {results['train_r2']:.4f}")
    print(f"  Val    — RMSE: {results['val_rmse']:.4f}, MAE: {results['val_mae']:.4f}, R²: {results['val_r2']:.4f}")
    print(f"  Test   — RMSE: {results['test_rmse']:.4f}, MAE: {results['test_mae']:.4f}, R²: {results['test_r2']:.4f}")
    return results

In [ ]:
# ── 1a: Linear Regression ──
print("Training Linear Regression...")
lr = LinearRegression(featuresCol="features", labelCol="label_reg", maxIter=100)
lr_model = lr.fit(train_df)
eval_regression("Linear Regression", lr_model, train_df, val_df, test_df)

lr_model.write().overwrite().save(os.path.join(MODELS_DIR, "linear_regression"))
print("Model saved.")

In [ ]:
# ── 1b: Decision Tree Regressor ──
print("Training Decision Tree Regressor...")
dt_reg = DecisionTreeRegressor(featuresCol="features", labelCol="label_reg", maxDepth=10)
dt_reg_model = dt_reg.fit(train_df)
eval_regression("Decision Tree Regressor", dt_reg_model, train_df, val_df, test_df)

dt_reg_model.write().overwrite().save(os.path.join(MODELS_DIR, "dt_regressor"))
print("Model saved.")

In [ ]:
# ── 1c: Random Forest Regressor (with CV tuning) ──
print("Training Random Forest Regressor with CrossValidator...")
rf_reg = RandomForestRegressor(featuresCol="features", labelCol="label_reg", seed=42)

paramGrid_rf = (
    ParamGridBuilder()
    .addGrid(rf_reg.numTrees, [50, 100])
    .addGrid(rf_reg.maxDepth, [8, 12])
    .build()
)

cv_rf = CrossValidator(
    estimator=rf_reg,
    estimatorParamMaps=paramGrid_rf,
    evaluator=reg_evaluator_rmse,
    numFolds=3,
    seed=42
)

cv_rf_model = cv_rf.fit(train_df)
rf_best = cv_rf_model.bestModel

print(f"Best RF: numTrees={rf_best.getNumTrees}, maxDepth={rf_best._java_obj.getMaxDepth()}")
eval_regression("Random Forest Regressor", rf_best, train_df, val_df, test_df)

rf_best.write().overwrite().save(os.path.join(MODELS_DIR, "rf_regressor"))
print("Model saved.")

In [ ]:
# ── 1d: Gradient Boosted Trees Regressor ──
print("Training GBT Regressor...")
gbt_reg = GBTRegressor(featuresCol="features", labelCol="label_reg",
                        maxIter=50, maxDepth=8, seed=42)
gbt_model = gbt_reg.fit(train_df)
eval_regression("GBT Regressor", gbt_model, train_df, val_df, test_df)

gbt_model.write().overwrite().save(os.path.join(MODELS_DIR, "gbt_regressor"))
print("Model saved.")

In [ ]:
# ── Regression Summary ──
import pandas as pd
reg_df = pd.DataFrame(regression_results)
print("\n" + "="*80)
print("REGRESSION RESULTS SUMMARY")
print("="*80)
print(reg_df.to_string(index=False))
reg_df.to_csv(os.path.join(RESULTS_DIR, "regression_results.csv"), index=False)

## 2. Classification Models (Predict Rating Tier)

In [ ]:
# ── Helper function ──
cls_evaluator_acc = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="accuracy")
cls_evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="f1")
cls_evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="weightedPrecision")
cls_evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_cls", metricName="weightedRecall")

classification_results = []

def eval_classification(model_name, model, train, val, test):
    results = {"model": model_name}
    for split_name, data in [("train", train), ("val", val), ("test", test)]:
        preds = model.transform(data)
        acc = cls_evaluator_acc.evaluate(preds)
        f1 = cls_evaluator_f1.evaluate(preds)
        prec = cls_evaluator_precision.evaluate(preds)
        rec = cls_evaluator_recall.evaluate(preds)
        results[f"{split_name}_accuracy"] = round(acc, 4)
        results[f"{split_name}_f1"] = round(f1, 4)
        results[f"{split_name}_precision"] = round(prec, 4)
        results[f"{split_name}_recall"] = round(rec, 4)
    classification_results.append(results)
    print(f"\n{model_name}:")
    print(f"  Train  — Acc: {results['train_accuracy']:.4f}, F1: {results['train_f1']:.4f}")
    print(f"  Val    — Acc: {results['val_accuracy']:.4f}, F1: {results['val_f1']:.4f}")
    print(f"  Test   — Acc: {results['test_accuracy']:.4f}, F1: {results['test_f1']:.4f}")
    return results

In [ ]:
# ── 2a: Logistic Regression ──
print("Training Logistic Regression (multiclass)...")
log_reg = LogisticRegression(featuresCol="features", labelCol="label_cls",
                              maxIter=100, family="multinomial")
log_reg_model = log_reg.fit(train_df)
eval_classification("Logistic Regression", log_reg_model, train_df, val_df, test_df)

log_reg_model.write().overwrite().save(os.path.join(MODELS_DIR, "logistic_regression"))
print("Model saved.")

In [ ]:
# ── 2b: Decision Tree Classifier ──
print("Training Decision Tree Classifier...")
dt_cls = DecisionTreeClassifier(featuresCol="features", labelCol="label_cls", maxDepth=10)
dt_cls_model = dt_cls.fit(train_df)
eval_classification("Decision Tree Classifier", dt_cls_model, train_df, val_df, test_df)

dt_cls_model.write().overwrite().save(os.path.join(MODELS_DIR, "dt_classifier"))
print("Model saved.")

In [ ]:
# ── 2c: Random Forest Classifier (with CV tuning) ──
print("Training Random Forest Classifier with CrossValidator...")
rf_cls = RandomForestClassifier(featuresCol="features", labelCol="label_cls", seed=42)

paramGrid_rf_cls = (
    ParamGridBuilder()
    .addGrid(rf_cls.numTrees, [50, 100])
    .addGrid(rf_cls.maxDepth, [8, 12])
    .build()
)

cv_rf_cls = CrossValidator(
    estimator=rf_cls,
    estimatorParamMaps=paramGrid_rf_cls,
    evaluator=cls_evaluator_f1,
    numFolds=3,
    seed=42
)

cv_rf_cls_model = cv_rf_cls.fit(train_df)
rf_cls_best = cv_rf_cls_model.bestModel

print(f"Best RF Classifier: numTrees={rf_cls_best.getNumTrees}, maxDepth={rf_cls_best._java_obj.getMaxDepth()}")
eval_classification("Random Forest Classifier", rf_cls_best, train_df, val_df, test_df)

rf_cls_best.write().overwrite().save(os.path.join(MODELS_DIR, "rf_classifier"))
print("Model saved.")

In [ ]:
# ── Classification Summary ──
cls_df = pd.DataFrame(classification_results)
print("\n" + "="*80)
print("CLASSIFICATION RESULTS SUMMARY")
print("="*80)
print(cls_df.to_string(index=False))
cls_df.to_csv(os.path.join(RESULTS_DIR, "classification_results.csv"), index=False)

## 3. K-Means Clustering (Title Segmentation)

In [ ]:
# ── Elbow method to find optimal K ──
print("Running Elbow Method for K-Means...")
silhouette_evaluator = ClusteringEvaluator(featuresCol="features")

k_values = range(2, 11)
silhouette_scores = []
wcss_scores = []

for k in k_values:
    kmeans = KMeans(featuresCol="features", k=k, seed=42, maxIter=20)
    km_model = kmeans.fit(train_df)
    preds = km_model.transform(train_df)
    sil = silhouette_evaluator.evaluate(preds)
    wcss = km_model.summary.trainingCost
    silhouette_scores.append(sil)
    wcss_scores.append(wcss)
    print(f"  K={k}: Silhouette={sil:.4f}, WCSS={wcss:.0f}")

# Find best K by silhouette
best_k = list(k_values)[np.argmax(silhouette_scores)]
print(f"\nBest K by Silhouette: {best_k}")

In [ ]:
# ── Plot Elbow & Silhouette ──
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot(list(k_values), wcss_scores, 'bo-', linewidth=2)
ax1.set_xlabel('K', fontsize=14)
ax1.set_ylabel('WCSS', fontsize=14)
ax1.set_title('Elbow Method', fontsize=16, fontweight='bold')

ax2.plot(list(k_values), silhouette_scores, 'ro-', linewidth=2)
ax2.axvline(best_k, color='green', linestyle='--', label=f'Best K={best_k}')
ax2.set_xlabel('K', fontsize=14)
ax2.set_ylabel('Silhouette Score', fontsize=14)
ax2.set_title('Silhouette Analysis', fontsize=16, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(os.path.join(BASE_DIR, 'outputs', 'plots'), '11_kmeans_elbow.png'))
plt.show()

In [ ]:
# ── Train final K-Means with best K ──
print(f"Training final K-Means with K={best_k}...")
kmeans_final = KMeans(featuresCol="features", k=best_k, seed=42, maxIter=50)
kmeans_model = kmeans_final.fit(train_df)

# Analyze clusters
clustered = kmeans_model.transform(df_ml)

print("\nCluster Analysis:")
cluster_stats = (
    clustered.groupBy("prediction")
    .agg(
        F.count("*").alias("count"),
        F.round(F.mean("label_reg"), 2).alias("avg_rating"),
        F.round(F.stddev("label_reg"), 2).alias("std_rating"),
        F.round(F.mean("runtimeMinutes"), 0).alias("avg_runtime"),
        F.round(F.mean("startYear"), 0).alias("avg_year"),
        F.round(F.mean("logNumVotes"), 2).alias("avg_log_votes")
    )
    .orderBy("prediction")
)
cluster_stats.show()

kmeans_model.write().overwrite().save(os.path.join(MODELS_DIR, "kmeans"))
print("K-Means model saved.")

In [ ]:
# ── Save clustering results ──
clustering_results = {
    "best_k": best_k,
    "silhouette_scores": {str(k): round(s, 4) for k, s in zip(k_values, silhouette_scores)},
    "wcss_scores": {str(k): round(w, 2) for k, w in zip(k_values, wcss_scores)}
}

with open(os.path.join(RESULTS_DIR, "clustering_results.json"), "w") as f:
    json.dump(clustering_results, f, indent=2)

print("All models trained and saved!")
print("\nProceed to 05_evaluation.ipynb")